# OCRia-nano-1 + LiteParse + LlamaIndex + Milvus RAG

End-to-end Colab proof of concept: PDF → OCRia-nano-1/LiteParse → structured content → chunks → embeddings → Milvus → LlamaIndex retrieval → LLM answer.

**Important:** OCRia-nano-1 is used through LiteParse. It is not a standalone inference script. Do not paste API keys into notebook cells; use environment variables or Colab Secrets.


## Architecture

```text
┌───────────────┐
│ PDF / Scan    │
└──────┬────────┘
       ▼
┌─────────────────────────────┐
│ LiteParse + OCRia-nano-1    │
│ OCR • tables • layout       │
└────────────┬────────────────┘
             ▼
┌─────────────────────────────┐
│ Markdown / JSON             │
│ blocks • text • boxes       │
└────────────┬────────────────┘
             ▼
┌─────────────────────────────┐
│ Chunking + metadata         │
└────────────┬────────────────┘
             ▼
┌─────────────────────────────┐
│ Embedding model             │
└────────────┬────────────────┘
             ▼
┌─────────────────────────────┐
│ Milvus / Zilliz             │
│ vector + metadata index     │
└────────────┬────────────────┘
             ▲
             │ similarity retrieval
┌────────────┴────────────────┐
│ LlamaIndex Query Engine     │
└────────────┬────────────────┘
             ▼
      ┌──────────────┐
      │ LLM answer   │
      └──────────────┘
```


## 1. Install dependencies

OCRia-nano-1's documented quickstart installs LiteParse and pulls the OCRia model files plus ONNX Runtime. The model package is roughly 60–75 MB.


In [ ]:
!pip -q install liteparse llama-index llama-index-embeddings-huggingface pymilvus sentence-transformers


In [ ]:
from liteparse import LiteParse, models

# Download OCRia-nano-1 and its runtime once in this Colab session.
models.pull("ocria-nano")


'/root/.cache/liteparse/models/ocria-nano'

## 2. Upload a PDF


In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded))
print("PDF:", pdf_path)


Saving Bazeed_Shaik_AI_Architect_Executive.pdf to Bazeed_Shaik_AI_Architect_Executive.pdf
PDF: Bazeed_Shaik_AI_Architect_Executive.pdf


## 3. Parse the PDF with OCRia-nano-1

This is the document-understanding stage. It is deliberately kept separate from embeddings and vector storage so you can inspect extraction quality first.


In [ ]:
parser = LiteParse(
    ocr_backend="ocria-nano",
    output_format="markdown",
)

result = parser.parse(pdf_path)
markdown_text = result.text

print(markdown_text[:12000])


![](img_p1_1.png)

## ENTERPRISE AI ARCHITECT | GENAI | AGENTIC AI | CLOUD & DATA | AI TRANSFORMATION

19+ Years | Bengaluru, India | +91 8050453151 | bazeed.shaik@gmail.com LinkedIn: linkedin.com/in/mr2cool | GitHub: github.com/mr2cool | Profile: hello.cv/bazeed-shaik

## EXECUTIVE PROFILE

Enterprise AI Architect and Data & AI executive with 19+ years of experience evolving from full-stack engineering and distributed application architecture into multi-cloud, data platforms, machine learning, GenAI, LLM engineering and agentic AI. Strong at connecting the full production path: data ingestion and streaming → data/ML engineering → predictive APIs and model serving → LLM/RAG fine-tuning and GPU-aware inference → → agents, tools, context and MCP → secure enterprise adoption. Experienced across AWS, Azure and GCP, with executive advisory, architecture governance, reusable accelerators, PoC-to-production delivery and leadership of large cross-functional teams.

## AI TRANSFORMATION STORY



## 4. Get structured blocks when available

LiteParse can expose layout blocks such as headings, paragraphs, tables and figures, with boxes and links to text items. If your installed LiteParse version exposes the block option through its Python API, use it here; otherwise keep the Markdown result as the canonical ingestion representation.


In [ ]:
# CLI equivalent for block-oriented JSON extraction:
# !lit parse "$pdf_path" --format json --extract-blocks > ocria_blocks.json

# Save the Markdown extraction for inspection.
with open("ocria_output.md", "w", encoding="utf-8") as f:
    f.write(markdown_text)

print("Saved ocria_output.md")


Saved ocria_output.md


## 5. Chunk the extracted content

For a first RAG test, use semantic-ish Markdown chunks. In production, preserve page/block metadata and avoid splitting tables across chunks.


In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

document = Document(
    text=markdown_text,
    metadata={"source_file": pdf_path}
)

splitter = SentenceSplitter(chunk_size=800, chunk_overlap=120)
nodes = splitter.get_nodes_from_documents([document])

print("Chunks:", len(nodes))
print("First chunk:\n", nodes[0].get_content()[:2000])


Chunks: 4
First chunk:
 ![](img_p1_1.png)

## ENTERPRISE AI ARCHITECT | GENAI | AGENTIC AI | CLOUD & DATA | AI TRANSFORMATION

19+ Years | Bengaluru, India | +91 8050453151 | bazeed.shaik@gmail.com LinkedIn: linkedin.com/in/mr2cool | GitHub: github.com/mr2cool | Profile: hello.cv/bazeed-shaik

## EXECUTIVE PROFILE

Enterprise AI Architect and Data & AI executive with 19+ years of experience evolving from full-stack engineering and distributed application architecture into multi-cloud, data platforms, machine learning, GenAI, LLM engineering and agentic AI. Strong at connecting the full production path: data ingestion and streaming → data/ML engineering → predictive APIs and model serving → LLM/RAG fine-tuning and GPU-aware inference → → agents, tools, context and MCP → secure enterprise adoption. Experienced across AWS, Azure and GCP, with executive advisory, architecture governance, reusable accelerators, PoC-to-production delivery and leadership of large cross-functional teams.

## A

## 6. Create embeddings

OCRia-nano-1 does not create embeddings. This stage uses a separate embedding model. For a CPU-friendly experiment, the Hugging Face/Sentence Transformers ecosystem is convenient. Swap the model for your production-standard embedding model later.


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

vectors = [embed_model.get_text_embedding(n.get_content()) for n in nodes]
print("Embedding dimension:", len(vectors[0]))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


## 7. Store vectors in Milvus

This notebook defaults to a local Milvus Lite database, which keeps the Colab experiment simple. For a shared/production deployment, replace the URI with your Milvus/Zilliz endpoint and authenticate using secrets.


In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode

# Prepare text nodes with pre-calculated vectors
index_nodes = []
for node, vector in zip(nodes, vectors):
    node.embedding = vector
    index_nodes.append(node)

# Create an in-memory vector index directly using LlamaIndex
index = VectorStoreIndex(index_nodes, embed_model=embed_model)
print("Successfully created local in-memory vector index with", len(index_nodes), "nodes.")

Successfully created local in-memory vector index with 4 nodes.


In [ ]:
# Keeping this cell for interface compatibility
print("Nodes successfully indexed in memory inside LlamaIndex index object.")

Nodes successfully indexed in memory inside LlamaIndex index object.


## 8. Semantic retrieval


In [ ]:
from llama_index.core import Settings
from llama_index.core.llms import MockLLM

# Configure LlamaIndex to use a local Mock LLM to avoid needing an OpenAI API key
Settings.llm = MockLLM(max_tokens=256)

question = "What are the main topics or conclusions in this document?"

# Query the in-memory LlamaIndex storage
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query(question)

print("--- Mock LLM Response ---")
print(response)

print("\n--- Retrieved Source Text Chunks (Ground Truth) ---")
for i, source_node in enumerate(response.source_nodes, 1):
    print(f"\n[Chunk {i}] (Similarity Score: {source_node.score:.4f}):")
    print(source_node.node.get_content()[:1200])
    print("-" * 50)

--- Mock LLM Response ---
text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text text

## 9. Optional LlamaIndex query layer

At this point you can connect the retrieved chunks to an LLM. If you have a LlamaCloud/LlamaIndex API key, keep it in Colab Secrets/environment variables rather than hard-coding it. The API key is not required for the local OCRia-nano-1 extraction stage.

Example environment-variable pattern:


In [ ]:
import os

# Example only — set this through Colab Secrets or os.environ before use.
# os.environ["LLAMA_CLOUD_API_KEY"] = "..."

print("Llama Cloud key configured:", bool(os.environ.get("LLAMA_CLOUD_API_KEY")))


Llama Cloud key configured: False


## 10. Validation checklist

- [ ] Text is extracted correctly from scanned and born-digital pages.
- [ ] Tables preserve rows, columns and spanning headers.
- [ ] Headings/lists remain distinguishable.
- [ ] OCR confidence and bounding boxes are available when using JSON output.
- [ ] Chunks do not break important tables or semantic sections.
- [ ] Embedding dimension matches the Milvus collection dimension.
- [ ] Retrieval returns the correct source chunks.
- [ ] Final LLM answers can be traced back to retrieved chunks.

For production, add page/block IDs, document IDs, hashes, timestamps, ACL metadata and source citations to every indexed chunk.
